In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import scipy.stats
from processing.satellites import gee_collections, get_bands, get_mean_value, process_gee_collection, plot_time_series, regenerate_all_plots, avdc_collections, download_avdc_omi_station_data

In [ ]:
# re-generate all plots of GEE satellite data, identify new collections
# NB: GEE collections are specified under satellites.py
collections_not_processed, errors = regenerate_all_plots(gee_collections=gee_collections)

# process specified collections
collections_subset = collections_not_processed
# collections_subset = ['MODIS/061/MOD08_M3', 'TRMM/3B42']
for collection in gee_collections.keys():
    if collection in collections_subset:
        for variable in gee_collections[collection]['variables']:
            process_gee_collection(collection=collection, variable=variable, verbosity=1)

In [ ]:
# download ready-made station data timeseries from AVDC
collection = 'OMI/V03/L2OVP/OMTO3'
product = 'aura_omi_l2ovp_omto3_v8.5_nairobi_175.txt'
df, metadata = download_avdc_omi_station_data(collection=collection, product=product)
display(metadata)
print(df.describe())

In [ ]:
# Read .parquets and plot timeseries and correlations of Aura, S5P and TOMS for the station NRB

# Copernicus S5P total ozone product
df_s5p = pl.read_parquet('data/level3/copernicus/s5p/offl/l3_o3/O3_column_number_density.parquet')
df_s5p.drop_in_place('mkn')
df_s5p = df_s5p.with_columns(pl.col('dte').dt.date())
df_s5p = df_s5p.rename({'nrb': 'tco_s5p'})
df_s5p = df_s5p.filter(pl.col('tco_s5p') > 0)
# unit conversion factor
# 1 DU = 2.687×10202.687×1020 molecules/m²; 1 mol = 6.022×10236.022×1023 molecules
# DU=mol/m2×(2.687×10206.022×1023​)≈mol/m2×2241.15
cf = 2241.15
df_s5p = df_s5p.with_columns(pl.col('tco_s5p')*cf)
print(df_s5p.describe())

# TOMS total ozone product
df_toms = pl.read_parquet('data/level3/toms/merged/ozone.parquet')
df_toms.drop_in_place('mkn')
df_toms = df_toms.rename({'nrb': 'tco_toms'})
df_toms = df_toms.with_columns(pl.col('dte').dt.date())
df_toms = df_toms.filter(pl.col('tco_toms') > 0)
print(df_toms.describe())

# Aura OMI total ozone product
df_aura = pl.read_parquet('data/level3/aura/aura_omi_l2ovp_omto3_v8.5_nairobi_175.parquet')
df_aura = df_aura.rename({'Ozone': 'tco_aura'})
# aggregate to daily values
df_aura = df_aura.group_by('dte', maintain_order=True).agg([pl.col('tco_aura').mean(),])
print(df_aura.describe())

In [ ]:
# plot timeseries
fig = plt.figure(figsize=(10, 7))
plt.plot(df_toms['dte'], df_toms['tco_toms'], c='orange', label='TOMS merged')
plt.plot(df_aura['dte'], df_aura['tco_aura'], c='b', label='AVDC EOS Aura OMI OMTO3 (v8.5, Collection 3)')
plt.plot(df_s5p['dte'], df_s5p['tco_s5p'], c='r', label=f'Copernicus S5P offl l3 O3')
plt.legend()
plt.title('NRB Total column ozone - daily values')
plt.ylabel ('Ozone (DU)')
plt.show()
fig.savefig('results/satellites/aura_vs_s5p_vs_toms_timeseries.png')

In [ ]:
# correlate the various satellite products
def lin_reg(x: list, y: list) -> dict:
    res = scipy.stats.linregress(x, y)
    return {'slope': res.slope, 'intercept': res.intercept, 'r2': res.rvalue**2, 'p': res.pvalue, 'std_err_slope': res.stderr, 
            'equation': f"f(x) = {res.slope:0.3f} x + {res.intercept:0.3f} (r2={res.rvalue**2:0.3f}, p={res.pvalue:0.3f}, se_slope={res.stderr:0.3f}, se_intercept={res.intercept_stderr:0.3f})"}

# combine data frames
df_aura_toms = df_toms.join(df_aura, on='dte', how='inner')
df_aura_toms.write_parquet('results/satellites/aura_vs_toms.parquet')
print(df_aura_toms.describe())
aura_toms = lin_reg(df_aura_toms['tco_toms'].to_list(), df_aura_toms['tco_aura'].to_list() )

df_s5p_toms = df_toms.join(df_s5p, on='dte', how='inner')
df_s5p_toms.write_parquet('results/satellites/s5p_vs_toms.parquet')
print(df_s5p_toms.describe())
s5p_toms = lin_reg(df_s5p_toms['tco_toms'].to_list(), df_s5p_toms['tco_s5p'].to_list())

# plot data and regression line
fig = plt.figure(figsize=(7, 7))
plt.scatter(df_aura_toms['tco_toms'], df_aura_toms['tco_aura'], c="orange", s=8, label=f'AVDC vs TOMS, n={len(df_aura_toms)}')
plt.scatter(df_s5p_toms['tco_toms'], df_s5p_toms['tco_s5p'], s=8, label=f'S5P vs TOMS, n={len(df_s5p_toms)}')

# plot regression lines
x = np.array(df_aura_toms['tco_toms'])
y_aura_toms = aura_toms['intercept'] + aura_toms['slope'] * x
plt.plot(x, y_aura_toms, color='red', label=aura_toms['equation'])
y_s5p_toms = s5p_toms['intercept'] + s5p_toms['slope'] * x
plt.plot(x, y_s5p_toms, color='blue', label=s5p_toms['equation'])

plt.legend()
plt.title('NRB Total column ozone - daily values')
plt.xlabel('TOMS total column ozone [DU]')
plt.ylabel('total column ozone [DU]')
plt.show()
fig.savefig('results/satellites/aura_vs_s5p_vs_toms_correlation.png')